In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import re

from tensorflow.keras.layers import Dense, SimpleRNN, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer

with open('train_data_true', 'r', encoding='utf-8') as f:
    text = f.read()
    text = text.replace('\ufeff', '')  # убираем первый невидимый символ
    text = re.sub(r'[^А-я ]', '', text)  # заменяем все символы кроме кириллицы на пустые символы

# парсим текст, как последовательность символов
num_characters = 34  # 33 буквы + пробел
tokenizer = Tokenizer(num_words=num_characters, char_level=True)  # токенизируем на уровне символов
tokenizer.fit_on_texts([text])  # формируем токены на основе частотности в нашем тексте
print(tokenizer.word_index)

inp_chars = 6
data = tokenizer.texts_to_matrix(text)  # преобразуем исходный текст в массив OHE
n = data.shape[0] - inp_chars  # так как мы предсказываем по трем символам - четвертый

X = np.array([data[i:i + inp_chars, :] for i in range(n)])
Y = data[inp_chars:]  # предсказание следующего символа

print(data.shape)

model = Sequential()
model.add(Input((inp_chars,
                 num_characters)))  # при тренировке в рекуррентные модели keras подается сразу вся последовательность, поэтому в input теперь два числа. 1-длина последовательности, 2-размер OHE
model.add(SimpleRNN(128, activation='tanh'))  # рекуррентный слой на 500 нейронов
model.add(Dense(num_characters, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')

history = model.fit(X, Y, batch_size=32, epochs=100)


def buildPhrase(inp_str, str_len=50):
    for i in range(str_len):
        x = []
        for j in range(i, i + inp_chars):
            x.append(tokenizer.texts_to_matrix(inp_str[j]))  # преобразуем символы в One-Hot-encoding

        x = np.array(x)
        inp = x.reshape(1, inp_chars, num_characters)

        pred = model.predict(inp)  # предсказываем OHE четвертого символа
        d = tokenizer.index_word[pred.argmax(axis=1)[0]]  # получаем ответ в символьном представлении

        inp_str += d  # дописываем строку

    return inp_str


res = buildPhrase("утренн")
print(res)

## Izah

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# os → əməliyyat sistemi ilə işləmək üçün modul.
#
# TF_CPP_MIN_LOG_LEVEL = '2' → TensorFlow-un artıq xəbərdarlıq və
# məlumat mesajlarını gizlədir (yalnız ciddi xətalar qalır).
# Modelin işinə təsiri yoxdur, sadəcə ekran təmiz görünür.
# DİQQƏT: Bu sətir TensorFlow import olunmazdan ƏVVƏL yazılmalıdır.


import numpy as np
# NumPy — massivlərlə işləmək üçün.
#
# Bu kodda:
# - X massivini yığmaq üçün (np.array)
# - giriş formasını dəyişmək üçün (reshape)


import re
# re — "regular expression" (requlyar ifadə) modulu.
# Mətni təmizləmək üçün istifadə olunur (aşağıda re.sub).


from tensorflow.keras.layers import Dense, SimpleRNN, Input
# Keras-dan üç qat import edirik.
#
# Dense:
# → Tam bağlı qat (MNIST kodundakı kimi).
#
# SimpleRNN:
# → YENİ QAT. Rekurrent (təkrarlanan) şəbəkə qatı.
#   Ardıcıllıqla işləyir və "yaddaşı" var: əvvəlki simvolları
#   xatırlayıb növbəti simvola təsir etdirir.
#   Conv2D şəkildə qonşu piksellərə baxırdı, SimpleRNN isə
#   mətndə ardıcıl simvollara baxır.
#
# Input:
# → Giriş formasını ayrıca qat kimi bildirir
#   (əvvəlki kodda InputLayer idi, bu da eyni işi görür).


from tensorflow.keras.models import Sequential
# Sequential model: qatlar ardıcıl şəkildə bir-birinin arxasınca işləyir.


from tensorflow.keras.optimizers import Adam
# Adam optimizatoru.
# DİQQƏT: Aşağıda istifadə olunmur, çünki compile-da
# optimizer='adam' sətir kimi yazılıb.


from tensorflow.keras.preprocessing.text import Tokenizer
# Tokenizer — mətni rəqəmlərə çevirən alət.
#
# Neyron şəbəkə hərfləri başa düşmür, yalnız ədədlər başa düşür.
# Tokenizer hər hərfə bir nömrə verir (məsələn, "а" → 1, "о" → 2).
# Bunu tərsinə də edə bilir: nömrədən hərfi tapır.


# =====================================================================
# BU KOD NƏDİR? — MƏTNİN DAVAMINI TƏXMİN EDƏN MODEL (text generation)
# =====================================================================
#
# Məqsəd: verilən 6 hərfə baxıb NÖVBƏTİ hərfi təxmin etmək,
# sonra bunu təkrarlaya-təkrarlaya mətn yazmaq.
#
# Ideya:
# 1. Mətni oxuyuruq, yalnız rus hərfləri və boşluq saxlayırıq.
# 2. Hər hərfi One-Hot formatına çeviririk.
# 3. 6 ardıcıl hərfi GİRİŞ veririk.
# 4. 7-ci hərfi DOĞRU CAVAB kimi veririk.
# 5. Model "bu 6 hərfdən sonra hansı hərf gəlir?" sualını öyrənir.
# 6. Sonra öz təxmin etdiyi hərfi sona əlavə edib prosesi təkrarlayır.
#
# ƏVVƏLKİ KODLARLA MÜQAYİSƏ:
#
# MNIST kodu:         şəkil → rəqəm (təsnifat)
# Rəngləndirmə kodu:  L kanalı → a, b kanalları (regressiya)
# BU KOD:             hərf ardıcıllığı → növbəti hərf (təsnifat)
#
# Yəni burada yenə təsnifat var (33 hərf + boşluq arasından seçirik),
# amma giriş şəkil deyil, ARDICILLIQDIR (sequence).


with open('train_data_true', 'r', encoding='utf-8') as f:
    text = f.read()
    text = text.replace('\ufeff', '')  # убираем первый невидимый символ
    text = re.sub(r'[^А-я ]', '', text)  # заменяем все символы кроме кириллицы на пустые символы
# Mətn faylını açıb oxuyur və təmizləyir.
#
# open('train_data_true', 'r', encoding='utf-8'):
# → 'train_data_true' adlı faylı oxu rejimində ('r') açır.
# → encoding='utf-8' → kirill hərflərini düzgün oxumaq üçün.
#
# text = f.read() → bütün faylı bir böyük sətir kimi oxuyur.
#
# text.replace('\ufeff', ''):
# → '\ufeff' görünməyən xüsusi simvoldur (BOM). Bəzi fayllar
#   başlanğıcda bunu saxlayır. Onu silirik ki, model bunu
#   "hərf" kimi öyrənməsin.
#
# re.sub(r'[^А-я ]', '', text):
# → [^А-я ] = "А-dan я-ya qədər hərflər və boşluq OLMAYAN hər şey".
# → Belə simvolları boş sətirlə əvəz edir (yəni silir).
# → Nəticədə mətndə yalnız kirill hərfləri və boşluq qalır.
#   Durğu işarələri, rəqəmlər, latın hərfləri silinir.
#
# DİQQƏT: Bu fayl kodla eyni qovluqda olmalıdır, yoxsa
# FileNotFoundError çıxar.
# Qeyd: 'А-я' aralığında 'ё' yoxdur (o ayrıca yerləşir), buna görə
# 'ё' hərfi silinir. Bəzi mətnlərdə bu kiçik fərq yarada bilər.


# parsим текст, как последовательность символов
num_characters = 34  # 33 буквы + пробел
# Mətndə neçə fərqli simvol ola biləcəyini bildirir: 34.
# (Rus əlifbasında 33 hərf + 1 boşluq = 34.)
#
# Bu rəqəm modelin giriş və çıxış ölçüsünü müəyyən edir.

tokenizer = Tokenizer(num_words=num_characters, char_level=True)  # токенизируем на уровне символов
# Tokenizer yaradır.
#
# char_level=True:
# → ƏN VACİB PARAMETR. Mətni SÖZLƏRƏ yox, HƏR HƏRFƏ bölür.
#   (Defolt halda Tokenizer sözlərlə işləyir.)
#
# num_words=num_characters:
# → Ən çox işlənən 34 simvolu nəzərə alır.
#
# DİQQƏT: Tokenizer nömrələməni 1-dən başlayır, 0 boş qalır.
# Buna görə OHE-nin ölçüsü əslində 35 olur (0..34).
# Amma num_words=34 olduğu üçün texts_to_matrix 34 sütun
# qaytarır (0..33). Bu səbəbdən ən az işlənən simvol "düşə" bilər.
# Əgər mətndə tam 34 fərqli simvol varsa, sonuncu itirilir.

tokenizer.fit_on_texts([text])  # формируем токены на основе частотности в нашем тексте
# Tokenizer-ə mətni göstərir ki, hansı hərflərin olduğunu öyrənsin.
#
# Hər hərfə nömrə verilir, ƏN ÇOX işlənən hərf → 1,
# sonrakı → 2 və s. (tezliyə görə sıralanır).
#
# [text] → siyahı içində verilir, çünki funksiya mətnlər
# siyahısı gözləyir (bizdə yalnız bir mətn var).

print(tokenizer.word_index)
# Hərf → nömrə lüğətini çap edir. Məsələn:
# {' ': 1, 'о': 2, 'е': 3, 'а': 4, ...}
#
# Burada boşluq adətən ən çox işlənən simvol olur (1-ci).


inp_chars = 6
# Modelə neçə hərf verəcəyimizi bildirir: 6.
#
# Yəni model hər dəfə 6 hərfə baxıb 7-ci hərfi təxmin edir.
# Bu rəqəmi artırsan, model daha uzun kontekstə baxar,
# amma SimpleRNN uzun ardıcıllıqda zəifdir (aşağıda izah olunur).

data = tokenizer.texts_to_matrix(text)  # преобразуем исходный текст в массив OHE
# Bütün mətni One-Hot Encoding (OHE) formatına çevirir.
#
# OHE nədir? Hər hərf uzunluğu 34 olan vektorla göstərilir,
# yalnız bir yerdə 1, qalanları 0.
#
# Məsələn, hərfin nömrəsi 3 olsa:
# [0, 0, 0, 1, 0, 0, ..., 0]
#           ↑ 3-cü mövqe
#
# MNIST kodunda etiketlər üçün to_categorical eyni işi görürdü.
#
# Nəticənin forması: (mətnin_uzunluğu, 34).
# Məsələn, mətn 10000 hərfdirsə → (10000, 34).

n = data.shape[0] - inp_chars  # так как мы предсказываем по трем символам - четвертый
# Neçə təlim nümunəsi alınacağını hesablayır.
#
# data.shape[0] → mətndəki ümumi hərf sayı.
# Sonuncu 6 hərfdən sonra təxmin ediləcək hərf qalmır,
# ona görə inp_chars çıxılır.
#
# Rus dilindəki şərh "3 simvola görə 4-cünü" deyir, amma
# indi inp_chars=6-dır, yəni 6-ya görə 7-ci.

X = np.array([data[i:i + inp_chars, :] for i in range(n)])
# "Sürüşən pəncərə" üsulu ilə giriş nümunələrini yığır.
#
# data[i:i+6, :] → i-ci hərfdən başlayaraq 6 ardıcıl hərf.
#
# Məsələn, mətn "утренний":
#   i=0 → "утрен"+"н"  → "утренн"
#   i=1 → "тренни"
#   i=2 → "ренний"
#
# Pəncərə hər dəfə 1 hərf sürüşür.
#
# Nəticənin forması: (n, 6, 34)
# n → nümunə sayı, 6 → ardıcıllıq uzunluğu, 34 → OHE ölçüsü.

Y = data[inp_chars:]  # предсказание следующего символа
# Düzgün cavablar: hər pəncərədən SONRA gələn hərf.
#
# data[6:] → 7-ci hərfdən sona qədər.
#   X[0] = hərflər 0..5   → Y[0] = hərf 6
#   X[1] = hərflər 1..6   → Y[1] = hərf 7
#
# Forma: (n, 34) — hər cavab OHE vektorudur.

print(data.shape)
# data-nın formasını çap edir, məsələn: (10000, 34).


model = Sequential()
# Boş ardıcıl model yaradır.

model.add(Input((inp_chars,
                 num_characters)))  
# при тренировке в рекуррентные модели keras подается сразу вся последовательность, 
# поэтому в input теперь два числа. 1-длина последовательности, 2-размер OHE
# Giriş qatı: forma (6, 34).
#
# İki rəqəm var (MNIST-də 28×28 idi, burada fərqlidir):
# → 6  → ardıcıllığın uzunluğu (neçə hərf)
# → 34 → hər hərfin OHE vektorunun ölçüsü
#
# Rus şərhi belə deyir: "Keras-ın rekurrent modellərinə təlim
# zamanı bütün ardıcıllıq birdən verilir, ona görə Input-da
# indi iki rəqəm var".

model.add(SimpleRNN(128, activation='tanh'))  # рекуррентный слой на 500 нейронов
# REKURRENT QAT. Bu kodun əsas hissəsi budur.
#
# 128 → neyron (yaddaş vahidi) sayı.
# (Rus şərhində 500 yazılıb, amma kodda 128-dir — şərh köhnədir,
#  kodun özü doğrudur.)
#
# Necə işləyir? 6 hərfi bir-bir oxuyur:
#
#   hərf1 → [RNN] → vəziyyət1
#   hərf2 + vəziyyət1 → [RNN] → vəziyyət2
#   hərf3 + vəziyyət2 → [RNN] → vəziyyət3
#   ...
#   hərf6 + vəziyyət5 → [RNN] → vəziyyət6
#
# "Vəziyyət" (hidden state) — şəbəkənin yaddaşıdır. Hər addımda
# əvvəl nə gördüyünü xatırlayır.
#
# Yalnız SONUNCU vəziyyət çıxışa verilir (return_sequences=False
# defoltdur). Yəni 6 hərfin hamısını "oxuyub xülasə çıxarır".
#
# activation='tanh' → qiymətləri -1...+1 aralığında saxlayır.
#   (RNN-lərdə relu əvəzinə adətən tanh istifadə olunur,
#    çünki tanh dəyərləri partlamaqdan qoruyur.)
#
# Forma: (6, 34) → (128,)
# Yəni 6×34 ölçülü ardıcıllıq 128 ədədlik vektora sıxılır.
#
# Parametr sayı:
#   giriş çəkiləri:   34 × 128 = 4352
#   yaddaş çəkiləri: 128 × 128 = 16384
#   bias:                       128
#   cəmi:                     20864

model.add(Dense(num_characters, activation='softmax'))
# Çıxış qatı.
#
# 34 neyron → hər hərf üçün bir ehtimal.
#
# activation='softmax' → 34 ədədi ehtimala çevirir (cəmi = 1).
# MNIST kodundakı kimi: ən yüksək ehtimallı sinif = təxmin.
#
# Forma: (128,) → (34,)
# Parametr: 128 × 34 + 34 = 4386

model.summary()
# Modelin quruluşunu və parametr sayını cədvəl kimi çap edir.
# Ümumi parametr: 20864 + 4386 = 25250.
#
# MNIST (~100 min) və rəngləndirmə (~3.9 milyon) kodları ilə
# müqayisədə bu model çox kiçikdir.


model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
# Öyrətmənin parametrlərini təyin edir.
#
# loss='categorical_crossentropy' → çoxsinifli təsnifat üçün
#   (MNIST kodundakı kimi). Burada siniflər 34 hərfdir.
#
# metrics=['accuracy'] → düzgün təxmin faizini göstərir.
#   Rəngləndirmə kodunda bu yox idi (regressiya idi),
#   burada isə var (təsnifatdır).
#
# optimizer='adam' → çəkiləri avtomatik yeniləyən optimizator.


history = model.fit(X, Y, batch_size=32, epochs=100)
# Öyrətməni başladır.
#
# batch_size=32 → hər addımda 32 nümunə.
# epochs=100 → bütün məlumat 100 dəfə keçilir.
#
# Rəngləndirmə kodunda cəmi 1 şəkil var idi, burada isə
# mətndən minlərlə pəncərə çıxır, ona görə real öyrənmə gedir.
#
# history → öyrətmə tarixçəsini (hər epoch-da loss və accuracy)
# saxlayır. İstəsən qrafik çəkə bilərsən.
#
# Qeyd: SimpleRNN qısa yaddaşlıdır. 6 hərf kontekst çox
# azdır, ona görə model məna daşıyan mətn yox, daha çox
# "rus dilinə bənzər hərf birləşmələri" yaradacaq.


def buildPhrase(inp_str, str_len=50):
    # Başlanğıc sətri götürüb ona str_len (50) hərf əlavə edən funksiya.
    # Hər addımda 1 hərf təxmin edib sona yazır.

    for i in range(str_len):
        # 50 dəfə təkrarlayır, hər dəfə bir yeni hərf.

        x = []
        for j in range(i, i + inp_chars):
            x.append(tokenizer.texts_to_matrix(inp_str[j]))  # преобразуем символы в One-Hot-encoding
        # Sətrin son 6 hərfini götürüb OHE-yə çevirir.
        #
        # range(i, i+6) → "pəncərə". i böyüdükcə pəncərə sürüşür,
        # çünki sətrin sonuna yeni hərflər əlavə olunur.
        #
        # i=0 → hərflər 0..5 (başlanğıc "утренн")
        # i=1 → hərflər 1..6 (6-cı hərf artıq modelin təxmin etdiyidir)
        #
        # texts_to_matrix(inp_str[j]) → bir hərfi (1, 34) formasında
        # OHE edir.

        x = np.array(x)
        # Siyahını massivə çevirir: forma (6, 1, 34).

        inp = x.reshape(1, inp_chars, num_characters)
        # Modelin gözlədiyi formaya salır: (1, 6, 34).
        #
        # 1 → batch ölçüsü (1 nümunə)
        # 6 → ardıcıllıq uzunluğu
        # 34 → OHE ölçüsü

        pred = model.predict(inp)  # предсказываем OHE четвертого символа
        # Model 34 hərfin ehtimallarını verir. Forma: (1, 34).
        # (Rus şərhində "dördüncü simvol" yazılıb, indi isə 7-cidir.)

        d = tokenizer.index_word[pred.argmax(axis=1)[0]]  # получаем ответ в символьном представлении
        # Ən yüksək ehtimallı hərfi tapır və simvola çevirir.
        #
        # pred.argmax(axis=1)[0] → ən böyük ehtimalın nömrəsi.
        # tokenizer.index_word → nömrədən hərfi tapan lüğət
        #   (word_index-in tərsi).
        #
        # DİQQƏT: argmax həmişə ƏN ehtimallı hərfi seçir (təsadüfilik yox).
        # Buna görə model eyni başlanğıca eyni nəticə verir və
        # bəzən təkrarlanan dövrə düşə bilər ("ааааа" kimi).

        inp_str += d  # дописываем строку
        # Təxmin edilən hərfi sətrin sonuna əlavə edir.
        # Növbəti dövrdə bu hərf də girişin bir hissəsi olacaq.

    return inp_str
    # Yekun sətri qaytarır: başlanğıc (6 hərf) + 50 yeni hərf.


res = buildPhrase("утренн")
# Başlanğıc kimi "утренн" verir. Dəqiq 6 hərf olmalıdır,
# çünki inp_chars=6.
#
# Model "утренн" → növbəti hərf (məsələn, "и") → "тренни" → ...
# beləcə 50 hərf yazır.

print(res)
# Nəticəni çap edir. Məsələn: "утренний свет ..." kimi bir şey.
# Mətnin keyfiyyəti təlim mətninin həcmindən və modelin
# gücündən asılıdır.